# 03 — Feature Engineering

Build derived features used in downstream correlation and cohort analysis:
- `age_group` (binned)
- `bio_length` and `bio_word_count` (from essay0)
- `total_essay_length` and `essays_written`
- `profile_completeness` (0-1 fraction of meaningful fields filled)
- Ordinal scales for `drinks`, `smokes`, `education`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import engineer_features

In [ ]:
df = pd.read_parquet('../data/processed/okcupid_clean.parquet')
df = engineer_features(df)
print(f"Loaded {len(df):,} rows, engineered features.")
df[['age', 'age_group', 'bio_length', 'bio_word_count', 'total_essay_length', 'essays_written', 'profile_completeness', 'drinks_score', 'smokes_score', 'education_score']].head()

## Distribution of new features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

df['age_group'].value_counts().sort_index().plot.bar(ax=axes[0,0], color='#4A90E2')
axes[0,0].set_title('Age group distribution')
axes[0,0].set_ylabel('Users')

df['bio_length'].clip(upper=2000).plot.hist(bins=50, ax=axes[0,1], color='#7ED321')
axes[0,1].set_title('Bio (essay0) length — clipped at 2000 chars')
axes[0,1].set_xlabel('Characters')

df['profile_completeness'].plot.hist(bins=20, ax=axes[1,0], color='#F5A623')
axes[1,0].set_title('Profile completeness score')
axes[1,0].set_xlabel('Fraction of fields filled')

df['essays_written'].value_counts().sort_index().plot.bar(ax=axes[1,1], color='#9013FE')
axes[1,1].set_title('Number of essays written (out of 10)')

plt.tight_layout()
plt.show()

In [ ]:
print('Profile completeness summary:')
print(df['profile_completeness'].describe().to_string())
print(f"\n% users with completeness >= 0.8: {(df['profile_completeness'] >= 0.8).mean():.1%}")
print(f"% users with completeness >= 0.5: {(df['profile_completeness'] >= 0.5).mean():.1%}")

## Save engineered dataset

In [ ]:
df.to_parquet('../data/processed/okcupid_features.parquet', index=False)
print('Saved to ../data/processed/okcupid_features.parquet')